# EdgeBPECoarsener example

This notebook builds repeated labeled paths, learns attachment-independent edge-BPE label rules, inspects the fitted rule symbols, and checks exact attachment-sensitive transform/decode behavior.


In [ ]:
from collections import Counter

from tree_coarsening import EdgeBPECoarsener
from tree_coarsening.utils import make_edge_bpe_dataset

X = make_edge_bpe_dataset(
    n_graphs=3,
    n_repeats=6,
    motif_labels=("A", "B", "C", "D"),
    seed=0,
)
G = X[0]
print(f"training graphs: {len(X)}")
print(f"first raw tree: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")


In [ ]:
raw_pair_counts = Counter(
    (G.nodes[parent]["label"], G.nodes[child]["label"])
    for parent, child in G.edges
)
for pair, count in raw_pair_counts.most_common():
    print(f"{pair}: {count}")


In [ ]:
coarsener = EdgeBPECoarsener(
    num_merges=3,
    min_pair_count=4,
).fit(X)

for step in coarsener.history_:
    print(
        f"rank={step['rank']} | "
        f"{step['parent_token']} -> {step['child_token']} | "
        f"raw count={step['count']} | contracted={step['actual_events']} | "
        f"new token={step['token']}"
    )


In [ ]:
print("Learned fitting symbols and coordinate specifications:")
for token, spec in coarsener.encoder_.vocab.symbols.items():
    if isinstance(token, tuple) and token and token[0] == "edge_bpe":
        print(f"{token}: site_count={spec.site_count}, root_count={spec.root_count}")


In [ ]:
H = coarsener.transform(G)
print(f"raw nodes: {G.number_of_nodes()}")
print(f"encoded nodes: {H.number_of_nodes()}")

for node, data in H.nodes(data=True):
    print(f"Node {node}: {data}")


In [ ]:
for parent, child, data in H.edges(data=True):
    print(f"Edge {parent} -> {child}: {data}")


In [ ]:
def uid_signature(graph):
    nodes = sorted(
        (data["uid"], data["label"], data["time"])
        for _, data in graph.nodes(data=True)
    )
    edges = sorted(
        (graph.nodes[parent]["uid"], graph.nodes[child]["uid"])
        for parent, child in graph.edges
    )
    return nodes, edges

G_roundtrip = coarsener.decode(H)
assert uid_signature(G_roundtrip) == uid_signature(G)
print("roundtrip ok")
